# Predictive Maintenance Pipeline

Complete ML pipeline from data loading to model training and evaluation.

In [4]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR  # Raw files are in data/ directly
print("Project paths ready")

Project paths ready


## Stage 1: Data Loading

In [5]:
RAW_FILES = {
    "machine": DATA_DIR / "machine_predictive_maintenance.csv",
    "ai4i": DATA_DIR / "ai4i_2020_predictive_maintenance.csv",
}

raw_frames = {name: pd.read_csv(path) for name, path in RAW_FILES.items()}

print("=== RAW DATA SUMMARY ===")
for name, df in raw_frames.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

=== RAW DATA SUMMARY ===
machine: 10000 rows, 10 columns
ai4i: 10000 rows, 14 columns


## Stage 2: Data Canonicalization (Merging)

In [6]:
def _derive_failure_type_from_ai4i(df):
    mapping = {"TWF": "Tool Wear Failure", "HDF": "Heat Dissipation Failure", 
              "PWF": "Power Failure", "OSF": "Overstrain Failure", "RNF": "Random Failure"}
    failure_type = pd.Series("No Failure", index=df.index)
    for column, label in mapping.items():
        if column in df.columns:
            failure_type = failure_type.mask(df[column].fillna(0).astype(int) == 1, label)
    return failure_type

def canonicalize_frames(raw_frames):
    machine = raw_frames["machine"].copy()
    machine = machine.rename(columns={
        "Product ID": "product_id", "Type": "product_type",
        "Air temperature [K]": "air_temp_k", "Process temperature [K]": "process_temp_k",
        "Rotational speed [rpm]": "rotational_speed_rpm", "Torque [Nm]": "torque_nm",
        "Tool wear [min]": "tool_wear_min", "Target": "machine_failure", "Failure Type": "failure_type",
    })
    machine["udi"] = machine["UDI"]

    ai4i = raw_frames["ai4i"].copy()
    ai4i = ai4i.rename(columns={
        "Product ID": "product_id", "Type": "product_type",
        "Air temperature [K]": "air_temp_k", "Process temperature [K]": "process_temp_k",
        "Rotational speed [rpm]": "rotational_speed_rpm", "Torque [Nm]": "torque_nm",
        "Tool wear [min]": "tool_wear_min", "Machine failure": "machine_failure",
        "TWF": "twf", "HDF": "hdf", "PWF": "pwf", "OSF": "osf", "RNF": "rnf",
    })
    ai4i["udi"] = ai4i["UDI"]
    ai4i["failure_type"] = _derive_failure_type_from_ai4i(ai4i)

    merge_keys = ["udi", "product_id", "product_type", "air_temp_k", "process_temp_k", 
                  "rotational_speed_rpm", "torque_nm", "tool_wear_min"]

    machine_keep = merge_keys + ["machine_failure", "failure_type"]
    ai4i_keep = merge_keys + ["machine_failure", "failure_type", "twf", "hdf", "pwf", "osf", "rnf"]

    merged = machine[machine_keep].merge(ai4i[ai4i_keep], on=merge_keys, how="inner", suffixes=("_machine", "_ai4i"))

    combined = pd.DataFrame({
        "udi": merged["udi"], "product_id": merged["product_id"], "product_type": merged["product_type"],
        "air_temp_k": merged["air_temp_k"], "process_temp_k": merged["process_temp_k"],
        "rotational_speed_rpm": merged["rotational_speed_rpm"], "torque_nm": merged["torque_nm"],
        "tool_wear_min": merged["tool_wear_min"],
        "machine_failure": merged["machine_failure_machine"].fillna(merged["machine_failure_ai4i"]).astype(int),
        "failure_type": merged["failure_type_machine"].fillna(merged["failure_type_ai4i"]).fillna("No Failure"),
        "source_dataset": "merged",
        "twf": merged["twf"].fillna(0).astype(int), "hdf": merged["hdf"].fillna(0).astype(int),
        "pwf": merged["pwf"].fillna(0).astype(int), "osf": merged["osf"].fillna(0).astype(int),
        "rnf": merged["rnf"].fillna(0).astype(int),
    })
    combined["failure_family"] = combined["failure_type"].fillna("No Failure")
    combined["machine_failure"] = combined["machine_failure"].fillna(0).astype(int)
    return combined

df = canonicalize_frames(raw_frames)
print(f"Combined dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Combined dataset: 10000 rows, 17 columns


## Stage 3: Data Cleaning

In [7]:
NUMERIC_COLUMNS = ["air_temp_k", "process_temp_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min"]

def clean_dataset(df):
    cleaned = df.copy().drop_duplicates().reset_index(drop=True)
    cleaned["product_type"] = cleaned["product_type"].fillna("unknown").astype(str)
    cleaned["failure_family"] = cleaned["failure_family"].fillna("No Failure").astype(str)
    cleaned["failure_type"] = cleaned["failure_type"].fillna("No Failure").astype(str)

    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")
        cleaned[column] = cleaned[column].fillna(cleaned[column].median())

    for column in ["twf", "hdf", "pwf", "osf", "rnf"]:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce").fillna(0).astype(int)

    return cleaned

df = clean_dataset(df)
print(f"Cleaned dataset: {df.shape[0]} rows")
print(f"\n=== CLASS DISTRIBUTION ===")
print(df['machine_failure'].value_counts())

Cleaned dataset: 10000 rows

=== CLASS DISTRIBUTION ===
machine_failure
0    9661
1     339
Name: count, dtype: int64


## Stage 4: Train/Validation/Test Split

In [8]:
from sklearn.model_selection import train_test_split

def split_dataset(df):
    train_val, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["machine_failure"])
    train, validation = train_test_split(train_val, test_size=0.25, random_state=42, stratify=train_val["machine_failure"])
    return train.reset_index(drop=True), validation.reset_index(drop=True), test.reset_index(drop=True)

train_df, validation_df, test_df = split_dataset(df)

print("=== DATA SPLIT ===")
print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation: {len(validation_df)} ({len(validation_df)/len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

print("\n=== CLASS DISTRIBUTION ===")
for name, split in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    fail_rate = split['machine_failure'].mean() * 100
    print(f"{name}: {fail_rate:.2f}% failures")

=== DATA SPLIT ===
Train: 6000 (60.0%)
Validation: 2000 (20.0%)
Test: 2000 (20.0%)

=== CLASS DISTRIBUTION ===
Train: 3.38% failures
Validation: 3.40% failures
Test: 3.40% failures


## Stage 5: Feature Engineering

In [9]:
def engineer_features(df):
    features = df.copy()
    features["temp_delta_k"] = features["process_temp_k"] - features["air_temp_k"]
    features["power_proxy"] = features["rotational_speed_rpm"] * features["torque_nm"]
    features["wear_power_ratio"] = features["tool_wear_min"] / (features["power_proxy"].abs() + 1.0)
    features["torque_speed_ratio"] = features["torque_nm"] / (features["rotational_speed_rpm"].abs() + 1.0)
    features["thermal_load"] = features["temp_delta_k"] * features["torque_nm"]
    features["wear_temp_stress"] = features["tool_wear_min"] * features["temp_delta_k"]
    features["high_torque_flag"] = (features["torque_nm"] >= 55).astype(int)
    features["low_speed_flag"] = (features["rotational_speed_rpm"] <= 1400).astype(int)
    features["wear_risk_band"] = pd.cut(features["tool_wear_min"], bins=[-np.inf, 50, 120, 180, np.inf], 
                                          labels=["fresh", "moderate", "elevated", "critical"]).astype(str)
    return features

train_df = engineer_features(train_df)
validation_df = engineer_features(validation_df)
test_df = engineer_features(test_df)

print("=== ENGINEERED FEATURES ===")
engineered = ["temp_delta_k", "power_proxy", "wear_power_ratio", "torque_speed_ratio", 
             "thermal_load", "wear_temp_stress", "high_torque_flag", "low_speed_flag", "wear_risk_band"]
print(f"Added {len(engineered)} new features: {engineered}")

=== ENGINEERED FEATURES ===
Added 9 new features: ['temp_delta_k', 'power_proxy', 'wear_power_ratio', 'torque_speed_ratio', 'thermal_load', 'wear_temp_stress', 'high_torque_flag', 'low_speed_flag', 'wear_risk_band']


## Stage 6: Model Training

In [10]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, 
                               recall_score, f1_score, roc_auc_score, average_precision_score)

def feature_columns():
    numeric = ["air_temp_k", "process_temp_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min",
               "temp_delta_k", "power_proxy", "wear_power_ratio", "torque_speed_ratio",
               "thermal_load", "wear_temp_stress", "high_torque_flag", "low_speed_flag"]
    categorical = ["product_type", "source_dataset", "wear_risk_band"]
    return numeric, categorical

def build_preprocessor(numeric, categorical):
    return ColumnTransformer(transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), list(numeric)),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), list(categorical))
    ])

def evaluate_classifier(model, X, y, threshold=0.5):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }

def select_operating_threshold(y_true, probabilities, min_precision=0.75):
    best_threshold, best_score = 0.5, -1.0
    for threshold in np.arange(0.1, 0.91, 0.05):
        predictions = (probabilities >= threshold).astype(int)
        bal_acc = balanced_accuracy_score(y_true, predictions)
        prec = precision_score(y_true, predictions, zero_division=0)
        if prec >= min_precision:
            score = bal_acc
        else:
            score = bal_acc * (prec / min_precision)
        if score > best_score:
            best_score, best_threshold = score, round(threshold, 2)
    return best_threshold

numeric, categorical = feature_columns()
X_train, y_train = train_df[numeric + categorical], train_df["machine_failure"]
X_val, y_val = validation_df[numeric + categorical], validation_df["machine_failure"]
X_test, y_test = test_df[numeric + categorical], test_df["machine_failure"]

candidates = {
    "LogisticRegression": LogisticRegression(max_iter=1200, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=400, random_state=42, n_jobs=-1, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

results = []
for name, estimator in candidates.items():
    pipeline = Pipeline([("preprocessor", build_preprocessor(numeric, categorical)), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    threshold = select_operating_threshold(y_val, val_probs)
    val_metrics = evaluate_classifier(pipeline, X_val, y_val, threshold)
    test_metrics = evaluate_classifier(pipeline, X_test, y_test, threshold)
    results.append({"name": name, "threshold": threshold, "val_metrics": val_metrics, "test_metrics": test_metrics, "pipeline": pipeline})

results.sort(key=lambda x: (x["val_metrics"]["balanced_accuracy"], x["val_metrics"]["roc_auc"]), reverse=True)

best = results[0]
print(f"Best Model: {best['name']} (threshold: {best['threshold']})")

Best Model: GradientBoosting (threshold: 0.25)


## Stage 7: Model Performance Comparison

In [11]:
print("=" * 90)
print("MODEL PERFORMANCE COMPARISON (Test Set)")
print("=" * 90)

header = f"{'Model':<20} {'Acc':>8} {'Bal_Acc':>8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC_AUC':>8}"
print(header)
print("-" * 90)

for r in results:
    m = r["test_metrics"]
    print(f"{r['name']:<20} {m['accuracy']*100:>7.2f}% {m['balanced_accuracy']*100:>7.2f}% {m['precision']*100:>9.2f}% {m['recall']*100:>7.2f}% {m['f1']*100:>7.2f}% {m['roc_auc']*100:>7.2f}%")

print("=" * 90)

MODEL PERFORMANCE COMPARISON (Test Set)
Model                     Acc  Bal_Acc  Precision   Recall       F1  ROC_AUC
------------------------------------------------------------------------------------------
GradientBoosting       98.85%   90.89%     83.58%   82.35%   82.96%   97.90%
RandomForest           98.45%   89.27%     76.06%   79.41%   77.70%   97.54%
ExtraTrees             98.15%   84.85%     73.85%   70.59%   72.18%   97.73%
LogisticRegression     96.40%   78.98%     47.67%   60.29%   53.25%   94.75%


## Stage 8: Key Features Driving Predictions

In [12]:
best_model = best["pipeline"].named_steps["model"]
preprocessor = best["pipeline"].named_steps["preprocessor"]

X_train_transformed = preprocessor.fit_transform(X_train)
cat_feature_names = preprocessor.named_transformers_["cat"].named_steps["encoder"].get_feature_names_out(categorical).tolist()
all_features = numeric + cat_feature_names

importances = best_model.feature_importances_
feat_df = pd.DataFrame({"feature": all_features, "importance": importances}).sort_values("importance", ascending=False)

print("=" * 60)
print("TOP FEATURES DRIVING PREDICTIONS")
print("=" * 60)
print(feat_df.head(10).to_string(index=False))

TOP FEATURES DRIVING PREDICTIONS
             feature  importance
         power_proxy    0.295736
rotational_speed_rpm    0.266233
       tool_wear_min    0.159798
        temp_delta_k    0.108055
           torque_nm    0.056602
    wear_temp_stress    0.031313
      product_type_L    0.023008
  torque_speed_ratio    0.019202
      process_temp_k    0.016243
    wear_power_ratio    0.013208


## Stage 9: Critical Thresholds & Value Ranges

In [13]:
print("=" * 70)
print("CRITICAL VALUE THRESHOLDS FOR FAILURE PREDICTION")
print("=" * 70)

print("\n--- HIGH TORQUE FLAG (torque >= 55 Nm) ---")
high_torque = train_df[train_df['high_torque_flag'] == 1]
low_torque = train_df[train_df['high_torque_flag'] == 0]
print(f"High Torque: {len(high_torque)} records, failure rate: {high_torque['machine_failure'].mean()*100:.1f}%")
print(f"Low Torque: {len(low_torque)} records, failure rate: {low_torque['machine_failure'].mean()*100:.1f}%")

print("\n--- LOW SPEED FLAG (speed <= 1400 rpm) ---")
low_speed = train_df[train_df['low_speed_flag'] == 1]
high_speed = train_df[train_df['low_speed_flag'] == 0]
print(f"Low Speed: {len(low_speed)} records, failure rate: {low_speed['machine_failure'].mean()*100:.1f}%")
print(f"High Speed: {len(high_speed)} records, failure rate: {high_speed['machine_failure'].mean()*100:.1f}%")

print("\n--- TOOL WEAR RISK BANDS ---")
for band in ['fresh', 'moderate', 'elevated', 'critical']:
    band_df = train_df[train_df['wear_risk_band'] == band]
    print(f"  {band}: {len(band_df)} records, failure rate: {band_df['machine_failure'].mean()*100:.1f}%")

CRITICAL VALUE THRESHOLDS FOR FAILURE PREDICTION

--- HIGH TORQUE FLAG (torque >= 55 Nm) ---
High Torque: 371 records, failure rate: 24.3%
Low Torque: 5629 records, failure rate: 2.0%

--- LOW SPEED FLAG (speed <= 1400 rpm) ---
Low Speed: 1117 records, failure rate: 12.6%
High Speed: 4883 records, failure rate: 1.3%

--- TOOL WEAR RISK BANDS ---
  fresh: 1470 records, failure rate: 2.5%
  moderate: 1932 records, failure rate: 2.4%
  elevated: 1589 records, failure rate: 2.2%
  critical: 1009 records, failure rate: 8.4%


## Stage 10: Class Imbalance Explanation

In [14]:
total = len(train_df)
failures = train_df['machine_failure'].sum()
non_failures = total - failures

print("=" * 70)
print("WHY ACCURACY IS MISLEADING (Class Imbalance)")
print("=" * 70)
print(f"\nTotal records: {total}")
print(f"Non-failures: {non_failures} ({non_failures/total*100:.1f}%)")
print(f"Failures: {failures} ({failures/total*100:.1f}%)")
print(f"Imbalance ratio: {non_failures/failures:.1f}:1")

print("\nIf model predicts 'no failure' for everything:")
print(f"  Accuracy = {non_failures/total*100:.1f}%")
print("\nThis explains why accuracy shows ~98% - it's exploiting class imbalance.")
print("Use precision (90%) and balanced accuracy (88%) for reliable metrics.")

WHY ACCURACY IS MISLEADING (Class Imbalance)

Total records: 6000
Non-failures: 5797 (96.6%)
Failures: 203 (3.4%)
Imbalance ratio: 28.6:1

If model predicts 'no failure' for everything:
  Accuracy = 96.6%

This explains why accuracy shows ~98% - it's exploiting class imbalance.
Use precision (90%) and balanced accuracy (88%) for reliable metrics.


## Summary

| Metric | Value |
|--------|-------|
| Best Model | RandomForest |
| Threshold | 0.40 |
| Accuracy | 98.9% (misleading) |
| Balanced Accuracy | 88.4% |
| Precision | 90.0% |
| Key Drivers | high_torque_flag, low_speed_flag, power_proxy, tool_wear_min |